# ERDCmetrics — Dataset Introduction

**Portal:** WARA data portal · **Dataset ID:** 1 · **Organization:** Ericsson · **Category:** metric

This notebook introduces the `ERDCmetrics` dataset: *server hard- and software
metrics data from the Ericsson Research Data Center (ERDC) in Lund*. It consists
of operational metrics gathered with Zabbix from the physical hardware and
software running in the data center: CPU, memory and disk utilization, network
interface counters, ICMP pings, temperatures, up-times, and more.

The dataset is organized as one file per day and per value type, covering
November 2022 to April 2025. Each row is a single measurement event: the host
that reported a measurement point (called an *item* in Zabbix), the time, and
the value. The dataset is meant for time-series analysis and monitoring of
server behavior, and for building models on operating states.

In this notebook we

* install the data portal client and load the data,
* inspect the dataset metadata and the file layout,
* go through the schema of the Zabbix history tables,
* look at the hosts and the items that are measured,
* pivot the events of one host into a per-minute time series,
* cluster the operating states of that server using PCA and t-SNE,
* collect the caveats and point to next steps.

## Contents

1. [Setup](#1-setup)
2. [Dataset metadata](#2-dataset-metadata)
3. [Available files](#3-available-files)
4. [Loading data and schema](#4-loading-data-and-schema)
5. [Quick look: hosts and items](#5-quick-look-hosts-and-items)
6. [From events to a time series](#6-from-events-to-a-time-series)
7. [Clustering the operating states](#7-clustering-the-operating-states)
8. [Caveats and next steps](#8-caveats-and-next-steps)

In [ ]:
%pip install -q git+https://github.com/wara-ops/DataportalClient.git@${DATAPORTAL_CLIENT_VERSION}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dataportal import DataportalClient

plt.rcParams['figure.figsize'] = [10, 6]

## 1. Setup

To access the data we use the data portal client together with a personal token:
paste your token inside the quotes in the cell below (see the *PortalBasics*
tutorial if you need to obtain one). If you are in the Jupyter environment
provided by the portal and the environment variable `DATAPORTAL_CLIENT_TOKEN` is
set, you can leave `token` empty instead.

In [ ]:
# paste your token inside the quotes (or leave empty if DATAPORTAL_CLIENT_TOKEN is set)
token = ''
dataset = 'ERDCmetrics'
client = DataportalClient(token)

## 2. Dataset metadata

The dataset record on the portal says:

> **ShortInfo:** Server hard- and software metrics data from the Ericsson Research Data Center in Lund.
> **LongInfo:** This data consists of operational metrics gathered from physical hardware and software running within ERDC using Zabbix. Each file covers roughly 1 day of data, and contains a table of either floats, ints or string as indicated in the filename. The files are packed into pickle files and compressed using bz2.

The full record, as returned by the portal (id, organization, category, tags, ...):

In [ ]:
client.fromDataset(dataset)
print(client.currentDataset().model_dump_json(indent=2))

Note that the files on the portal are now parquet files (the pickle/bz2 format
mentioned in the description was the older one); the portal client handles the
format automatically.

## 3. Available files

The dataset consists of one file per day and per value type: the file names
follow the pattern `history[_<type>]_<start>_<stop>_<n entries>.zst.parquet`,
where `<type>` is `float`, `uint`, `str`, or `text`. The cell below fetches the
full list of files (about 33,000; it takes roughly a minute) and summarizes it.

In [ ]:
files = pd.DataFrame(list(client.fromDataset(dataset).listFiles(limit=40000)))
files.head()

In [ ]:
files.assign(day=files.StartDate.str[:10]).groupby('MetricType').agg(
    files=('FileID', 'size'), first_day=('day', 'min'), last_day=('day', 'max'))

## 4. Loading data and schema

Let's load the files of the first day of the dataset (2022-11-16), one per
value type, using the `FileID`s from the list above.

In [ ]:
file_ids = {'float': 1, 'str': 112, 'uint': 331}
frames = {}
for mtype, fid in file_ids.items():
    frames[mtype] = client.getData(fid)
    print(f'{mtype}: {client.currentLoadedFile().MFileName}, {frames[mtype].shape}')

All files share the same schema: a Zabbix *history* table with one row per
measurement event.

| Column | Content |
|--------|---------|
| `hostid`, `host` | the host (server) that reported the event |
| `itemid`, `name` | the measurement point (item) and its human-readable name |
| `units` | the unit of the value, if it has one |
| `status`, `item_status` | Zabbix status flags (0 = normal/enabled) |
| `value_type` | the type of the value: 0 float, 1 character, 3 unsigned int, 4 text |
| `clock`, `ns` | the event time: unix seconds and nanoseconds |
| `value` | the measurement itself; the type depends on the file (float, string, or int) |

The value type is also part of the file name, and it decides the type of the
`value` column. The `float` and `uint` files hold the numeric metrics; the
`str` and `text` files hold occasional non-numeric values (checksums, system
descriptions, package lists, ...).

In [ ]:
frames['float'].head(3)

## 5. Quick look: hosts and items

Who is measured, and what? The tables below count the events per host across
the three loaded files, and the number of distinct items per file.

In [ ]:
all_events = pd.concat(frames.values())
print(f'{all_events.host.nunique()} hosts, {len(all_events)} events in one day')
for mtype, d in frames.items():
    print(f'{mtype}: {d.name.nunique()} distinct items')
print()
all_events.host.value_counts().head(12)

The ERDC runs some thirty-six servers (`eselda*` hosts), and the Zabbix server
itself is monitored like any other host. The hosts are not homogeneous: they
report different items, at different rates. The most frequently reported
items in the `float` file are the classic per-minute CPU, memory, and disk
metrics:

In [ ]:
frames['float'].name.value_counts().head(10)

The `str` file is much sparser and holds non-numeric values, for example
checksums of system files and SNMP system descriptions. It will be set aside
for the rest of this notebook, but it is useful for configuration-drift
analysis.

In [ ]:
frames['str'].name.value_counts()

## 6. From events to a time series

The raw data is a long table of events, which is not a convenient format for
time-series analysis. The standard transformation is to pick one host, group
the events per minute, and pivot the items into columns, so that each row is
one minute and each column is one measurement point. When several events of the
same item fall in one minute they are averaged, and minutes where an item was
not reported become `NaN`.

In [ ]:
# one host: eselda06u13
host = 'eselda06u13'
df_raw = pd.concat([frames['float'], frames['uint']])
df = df_raw.loc[df_raw.host == host].sort_values(['clock', 'ns'])
df['clock'] = pd.to_datetime(df.clock, unit='s').dt.floor('min')
df = df.loc[:, ['name', 'clock', 'value']]

df = df.pivot_table(index='clock', columns='name')
df.columns = df.columns.get_level_values(1)  # (value, name) -> name

# keep only the items that were reported at least half of the day
df = df.loc[:, df.isna().sum() < df.shape[0] / 2]
print(f'{df.shape[0]} minutes, {df.shape[1]} measurement points, '
      f'NaNs: {df.isna().sum().sum()}')
df.head(3)

The result is a tidy time series: 1440 minutes by 121 measurement points, with
no missing values left after dropping the sparsely reported items. Let's look
at a few of the columns over the first six hours.

In [ ]:
t0 = df.index[0]
tf = t0 + pd.Timedelta(6, unit='h')
sub = df.loc[t0:tf]
mps = ['Load average (1m avg)', 'Available memory in %', 'sda: Disk write rate']
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 6))
for ax, mp in zip(axes, mps):
    ax.plot(sub.index, sub[mp])
    ax.set_title(mp)
plt.tight_layout()
plt.show()

## 7. Clustering the operating states

The operating state of a server is high-dimensional: 121 measurement points per
minute. Let's start with [principal component analysis
(PCA)](https://en.wikipedia.org/wiki/Principal_component_analysis) to see how
much of the variance is carried by a few directions.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn import cluster

X = StandardScaler().fit_transform(df)

pca = PCA()
pca.fit(X)
plt.figure()
plt.stem(pca.explained_variance_)
plt.title('Explained variance per principal component')
plt.xlabel('component')
plt.xticks(range(1, 21))
plt.tight_layout()
plt.show()

The variance is spread over many components: the largest explains only about
11% of the total variance, and roughly 40 components are needed to capture 90%.
This is typical for raw operating metrics: many subsystems (CPU, memory,
disks, network) are only weakly correlated with each other.

We keep 15 components (still about 60% of the variance) and project the minutes
to 2 dimensions with [t-distributed stochastic neighbor embedding
(t-SNE)](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding):

In [ ]:
X15 = PCA(n_components=15).fit_transform(X)
X2 = TSNE(random_state=8).fit_transform(X15)

plt.figure()
plt.plot(X2[:, 0], X2[:, 1], 'C0.', ms=2)
plt.title('t-SNE projection of the 1440 minutes of the day')
plt.tight_layout()
plt.show()

The 2D projection is a visualization aid only: t-SNE distorts distances and can
make a structure look more distinct than it is. To actually label the minutes
we run a clustering algorithm on the 15-component data. Three options from the
[scikit-learn clustering module](https://scikit-learn.org/stable/modules/clustering.html):

In [ ]:
#clustering = cluster.DBSCAN(eps=5, min_samples=2).fit(X15)
#clustering = cluster.KMeans(n_clusters=8, n_init='auto', random_state=8).fit(X15)
clustering = cluster.AgglomerativeClustering(n_clusters=5).fit(X15)

# Sort the labels so that cluster 0 is the largest one (-1 = noise, if any).
labels, counts = np.unique(clustering.labels_, return_counts=True)
order = sorted(zip(labels, counts), key=lambda x: x[1], reverse=True)
cluster_labels = np.full(len(clustering.labels_), -1, dtype=int)
for new, (old, _) in enumerate(order):
    if old == -1:
        cluster_labels[clustering.labels_ == -1] = -1
    else:
        cluster_labels[clustering.labels_ == old] = new

# Plot the cluster labels on the t-SNE projection.
fig, ax = plt.subplots()
for c in sorted(set(cluster_labels)):
    if c == -1:
        continue
    ax.scatter(X2[cluster_labels == c, 0], X2[cluster_labels == c, 1],
               s=4, label=f'cluster {c}')
ax.legend()
ax.set_title('Cluster labels on the t-SNE projection')
plt.tight_layout()
plt.show()

With k=5 for one day of one server, most of the minutes fall into two large,
similar idle clusters; the smaller ones are busy periods and short bursts.
A fixed-k algorithm will find k clusters even when the data does not contain
them, so take the result with a grain of salt.

Which metrics make the clusters different? The table below lists, for each
cluster, the measurement points whose cluster mean deviates the most from the
daily average (ranked by the absolute z-score).

In [ ]:
df_norm = (df - df.mean()) / df.std()
rows = []
for c in sorted(set(cluster_labels)):
    if c == -1:
        continue
    sub = df.iloc[cluster_labels == c]
    dev = df_norm.iloc[cluster_labels == c].mean().abs().sort_values(ascending=False)
    for col in dev.head(4).index:
        rows.append({'cluster': c, 'minutes': len(sub), 'metric': col,
                     'z-score': round(dev[col], 2),
                     'cluster mean': round(sub[col].mean(), 3)})
pd.DataFrame(rows).set_index(['cluster', 'metric'])

The two large clusters differ mainly in slowly drifting quantities (uptime,
the Zabbix server's internal counters); the busy cluster shows clearly elevated
CPU utilization and load average, and the two small ones are short bursts of
disk activity on `sda` and `sdb`. Note that near-constant columns (like
uptime) can also top this ranking because their standard deviation is small.

Finally, here is the cluster label over the day, together with the CPU
utilization:

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 5),
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.index, df['CPU utilization'], 'k-', lw=0.5)
ax1.set_ylabel('CPU utilization (%)')
ax1.set_title('CPU utilization and cluster label over the day')
ax2.step(df.index, cluster_labels, where='mid', color='tab:blue', lw=1)
ax2.set_yticks([c for c in sorted(set(cluster_labels)) if c >= 0])
ax2.set_ylabel('cluster')
plt.tight_layout()
plt.show()

## 8. Caveats and next steps

**Caveats**

* **One row per event, not per minute.** The raw tables must be aggregated
  (as in section 6) to get time series; items that were not reported in a
  minute become `NaN`.
* **One file per day.** Long-term analysis requires loading many files (there
  are 33,000 of them). Use the `startDate`/`endDate` arguments of `listFiles`
  to pick the days you need, and the `entries`/`generator` arguments of
  `getData` to stream large files.
* **Heterogeneous hosts.** Different hosts report different items, at different
  rates; the `Zabbix server` host is a meta-host that reports about the
  monitoring system itself.
* **The `str` and `text` files are sparse and non-numeric** (checksums, system
  descriptions, package lists); they are more useful for configuration-drift
  analysis than for time series.
* **Counters and rates.** Some items (interface byte/packet counts, units
  `bps`) are counters or rates that need differencing or rescaling before they
  can be compared.
* **Units vary** (`%`, `s`, `bps`, `°C`, ...) and can be missing, and some item
  names are terse or non-obvious.

**Next steps**

* load several days and study the trends and anomalies over weeks or months,
* build monitoring dashboards from the pivoted time series,
* detect incidents: alert on excursions of the CPU, disk, and memory metrics,
  or cluster the operating states over many days,
* correlate the metrics of the different hosts (for example the Zabbix server
  load against the rate of the reported events),
* use the `str`/`text` files to track configuration drift (package lists, file
  checksums).